In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("datasets/unique_tech_classroom_air_quality.csv")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp                500 non-null    str    
 1   school_period            500 non-null    str    
 2   student_count_estimated  500 non-null    int64  
 3   co2_ppm                  500 non-null    float64
 4   pm25_ugm3                500 non-null    float64
 5   temperature_c            500 non-null    float64
 6   humidity_pct             500 non-null    float64
 7   robot_x_pos              500 non-null    float64
 8   robot_y_pos              500 non-null    float64
 9   ventilation_decision     500 non-null    str    
 10  air_quality_label        500 non-null    str    
dtypes: float64(6), int64(1), str(4)
memory usage: 43.1 KB


In [5]:
df.isnull().sum()

timestamp                  0
school_period              0
student_count_estimated    0
co2_ppm                    0
pm25_ugm3                  0
temperature_c              0
humidity_pct               0
robot_x_pos                0
robot_y_pos                0
ventilation_decision       0
air_quality_label          0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.describe()


,student_count_estimated,co2_ppm,pm25_ugm3,temperature_c,humidity_pct,robot_x_pos,robot_y_pos
count,500.000000,500.000000,500.000000,500.00000,500.000000,500.000000,500.000000
mean,11.184000,619.266400,8.439960,20.48980,44.594400,2.005800,1.990000
std,9.446301,155.967082,2.718404,0.19191,3.057506,1.415639,1.420781
min,0.000000,432.200000,4.400000,20.20000,41.300000,0.000000,0.000000
25%,3.000000,493.825000,6.170000,20.40000,41.800000,0.600000,0.600000
50%,7.000000,529.200000,7.080000,20.50000,42.800000,2.050000,2.000000
75%,22.000000,801.425000,11.512500,20.50000,47.800000,3.400000,3.400000
max,27.000000,893.000000,14.500000,21.00000,49.700000,4.000000,4.000000


In [8]:
import pandas as pd

def detect_potential_targets(df):
    """
    Identifies potential target variables based on cardinality, 
    data types, and column positioning.
    """
    potential_targets = []
    total_rows = len(df)
    
    for i, col in enumerate(df.columns):
        unique_values = df[col].nunique()
        dtype = df[col].dtype
        
        # Heuristics:
        # 1. Low cardinality (classification) but not a constant
        # 2. Often found in the last 2 columns of the dataset
        # 3. Naming conventions like 'label', 'target', 'decision', or 'class'
        
        is_low_cardinality = 1 < unique_values < (total_rows * 0.05)
        is_at_end = i >= (len(df.columns) - 2)
        has_target_name = any(word in col.lower() for word in ['label', 'decision', 'target', 'class', 'status'])
        
        if (is_low_cardinality and (is_at_end or has_target_name)):
            potential_targets.append({
                'column': col,
                'type': 'Classification' if dtype == 'object' or unique_values < 10 else 'Regression/Multi-class',
                'reason': 'Low cardinality or naming convention'
            })
            
    return potential_targets

# Example usage with the Classroom Air Quality schema:
# targets = detect_potential_targets(air_quality_df)

In [9]:
detect_potential_targets(df)

[{'column': 'ventilation_decision',
  'type': 'Classification',
  'reason': 'Low cardinality or naming convention'},
 {'column': 'air_quality_label',
  'type': 'Classification',
  'reason': 'Low cardinality or naming convention'}]

In [12]:
import pandas as pd
from pandas.api.types import (
    is_object_dtype,
    is_categorical_dtype,
    is_bool_dtype
)

TARGET_KEYWORDS = {"label", "decision", "target", "class", "status"}

def detect_potential_targets(
    df: pd.DataFrame,
    cardinality_threshold: float = 0.05,
    max_class_threshold: int = 20,
    end_column_window: int = 2
):
    """
    Detect potential target columns using optimized heuristics.

    Heuristics:
    1. Low cardinality relative to dataset size
    2. Target-like column names
    3. Columns near the end of dataset
    4. Datatype-aware classification

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe

    cardinality_threshold : float
        Max unique ratio for classification targets

    max_class_threshold : int
        Absolute upper bound for classification classes

    end_column_window : int
        Number of columns from end considered likely targets

    Returns
    -------
    list[dict]
        Ranked list of potential targets
    """

    total_rows = len(df)

    if total_rows == 0:
        return []

    # Compute nunique once for all columns (faster)
    nunique_series = df.nunique(dropna=False)

    potential_targets = []

    last_columns = set(df.columns[-end_column_window:])

    for col in df.columns:

        unique_values = nunique_series[col]

        # Skip constant columns
        if unique_values <= 1:
            continue

        dtype = df[col].dtype
        col_lower = col.lower()

        # Heuristics
        unique_ratio = unique_values / total_rows

        is_low_cardinality = (
            unique_ratio <= cardinality_threshold
            or unique_values <= max_class_threshold
        )

        has_target_name = any(
            keyword in col_lower
            for keyword in TARGET_KEYWORDS
        )

        is_at_end = col in last_columns

        # Combined scoring system
        score = (
            (3 if has_target_name else 0) +
            (2 if is_at_end else 0) +
            (2 if is_low_cardinality else 0)
        )

        # Ignore weak candidates
        if score < 5:
            continue

        # Determine ML problem type
        if (
            is_object_dtype(dtype)
            or is_categorical_dtype(dtype)
            or is_bool_dtype(dtype)
            or unique_values <= max_class_threshold
        ):
            target_type = (
                "Binary Classification"
                if unique_values == 2
                else "Multi-class Classification"
            )
        else:
            target_type = "Regression"

        potential_targets.append({
            "column": col,
            "target_type": target_type,
            "unique_values": int(unique_values),
            "unique_ratio": round(unique_ratio, 4),
            "score": score,
            "reasons": {
                "low_cardinality": is_low_cardinality,
                "target_name": has_target_name,
                "near_dataset_end": is_at_end
            }
        })

    # Sort by confidence score
    potential_targets.sort(
        key=lambda x: (-x["score"], x["unique_values"])
    )

    return potential_targets

In [13]:
targets = detect_potential_targets(df)

for t in targets:
    print(t)

{'column': 'ventilation_decision', 'target_type': 'Binary Classification', 'unique_values': 2, 'unique_ratio': np.float64(0.004), 'score': 7, 'reasons': {'low_cardinality': np.True_, 'target_name': True, 'near_dataset_end': True}}
{'column': 'air_quality_label', 'target_type': 'Binary Classification', 'unique_values': 2, 'unique_ratio': np.float64(0.004), 'score': 7, 'reasons': {'low_cardinality': np.True_, 'target_name': True, 'near_dataset_end': True}}


C:\Users\Rohit Shere\AppData\Local\Temp\ipykernel_5252\4093278813.py:97: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  or is_categorical_dtype(dtype)


In [ ]:
from agents.data_understanding_agent import load_dataset, generate_statistical_summary,get_basic_info, detect_column_types, detect_target_column
from model.llm import get_llm
from prompts.data_understanding import get_target_column_prompt

df = load_dataset("datasets/unique_tech_classroom_air_quality.csv")

statstical_summary = generate_statistical_summary(df)

column_types = detect_column_types(df)

llm = get_llm()

prompt = get_target_column_prompt(statstical_summary, column_types)

response = llm.invoke(prompt)



Unexpected argument 'stream' provided to ChatGoogleGenerativeAI. Did you mean: 'streaming'?


✅ Dataset Loaded Successfully


C:\Users\Rohit Shere\AppData\Local\Temp\ipykernel_10644\1999296887.py:11: UserWarning: WARNING! stream is not default parameter.
                stream was transferred to model_kwargs.
                Please confirm that stream is what you intended.
  llm = get_llm()


NameError: name 'statistical_summary' is not defined